# Part 1 — Markov Context Sufficiency
## Full reproduction + Bayes-optimal truncated-context baseline

This Jupyter notebook rebuilds the experimental core of the manuscript and adds the new information-theoretic decomposition:

\[
L_{model}-H^*=(H_L-H^*)+(L_{model}-H_L).
\]

It includes:
- main recovery curves for V=8, k={2,3,4}, 5 seeds,
- H* and H_L estimation,
- phase diagram,
- capacity control,
- V=32 robustness,
- sparse-kernel robustness,
- checkpoint/resume,
- exact kernel persistence,
- per-run DONE markers,
- CSV/JSON exports and PNG/PDF figures.

The notebook automatically creates `Markov_Context_Paper/Part1/` inside the Jupyter working directory and saves all Part 1 outputs there.


In [ ]:

import os, sys, json, math, time, random, hashlib, platform
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_BF16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))
print("AMP:", "BF16" if USE_BF16 else ("FP16" if torch.cuda.is_available() else "OFF"))


In [ ]:

# IMPORTANT: change this to persistent storage on the university server.
PROJECT_ROOT = (Path.cwd() / "Markov_Context_Paper" / "Part1").resolve()

RUNS_DIR = PROJECT_ROOT / "runs"
EXPORTS_DIR = PROJECT_ROOT / "exports"
CSV_DIR = EXPORTS_DIR / "csv"
FIG_DIR = EXPORTS_DIR / "figures"
THEORY_DIR = PROJECT_ROOT / "theory_cache"

for p in [PROJECT_ROOT, RUNS_DIR, EXPORTS_DIR, CSV_DIR, FIG_DIR, THEORY_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)


In [ ]:

# Experiment switches
QUICK_TEST = False

RUN_MAIN = True
RUN_CAPACITY = False
RUN_VOCAB32 = False
RUN_VOCAB32_K3_EXT = False
RUN_SPARSE = False

MAIN_V = 8
MAIN_KS = [2, 3, 4]
SEEDS = [0, 1, 2, 3, 4]
EXTRA_CONTEXT = 3

BASE_STEPS = 3000
BASE_BATCH = 1024
BASE_DMODEL = 256
BASE_LAYERS = 3
BASE_HEADS = 8
BASE_FF = 512
BASE_LR = 1e-3
WEIGHT_DECAY = 1e-2
SAVE_EVERY = 500
EVAL_BATCH = 4096
CHAIN_WARMUP = 64
CAPACITY_STEPS = 12000

THEORY_SAMPLES = 500_000
THEORY_BATCH = 4096

if QUICK_TEST:
    SEEDS = [0]
    BASE_STEPS = 80
    CAPACITY_STEPS = 120
    BASE_BATCH = 128
    EVAL_BATCH = 512
    SAVE_EVERY = 20
    THEORY_SAMPLES = 20_000
    THEORY_BATCH = 1024

print("QUICK_TEST =", QUICK_TEST)


In [ ]:

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def get_rng_state():
    out = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch_cpu": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        out["torch_cuda"] = torch.cuda.get_rng_state_all()
    return out

def set_rng_state(state):
    if state is None:
        return
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch_cpu"])
    if torch.cuda.is_available() and "torch_cuda" in state:
        torch.cuda.set_rng_state_all(state["torch_cuda"])

def safe_json(obj):
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    return obj

def save_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=safe_json)

def stable_hash(d, n=12):
    s = json.dumps(d, sort_keys=True, default=safe_json).encode()
    return hashlib.sha256(s).hexdigest()[:n]

save_json(PROJECT_ROOT / "environment.json", {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "bf16": USE_BF16,
    "numpy": np.__version__,
    "pandas": pd.__version__,
})


## Markov kernel and persistent chain sampler

Dense kernels use Gaussian logits + softmax. Sparse kernels retain only `s` active transitions per full context. The exact kernel is saved for every run.


In [ ]:

def context_to_index(context, V):
    idx = torch.zeros(context.shape[:-1], dtype=torch.long, device=context.device)
    for j in range(context.shape[-1]):
        idx = idx * V + context[..., j]
    return idx

def generate_kernel(V, k, seed, sparse_s=None, device=DEVICE):
    g = torch.Generator(device=device)
    g.manual_seed(seed + 10_000_019)
    n_states = V ** k
    logits = torch.randn((n_states, V), generator=g, device=device, dtype=torch.float32)
    if sparse_s is None:
        return torch.softmax(logits, dim=-1)
    scores = torch.rand((n_states, V), generator=g, device=device)
    active = scores.topk(sparse_s, dim=-1).indices
    masked = torch.full_like(logits, float("-inf"))
    masked.scatter_(1, active, logits.gather(1, active))
    return torch.softmax(masked, dim=-1)

class PersistentMarkovSampler:
    def __init__(self, P, V, k, batch_size, max_history, seed, warmup_steps=64):
        self.P = P
        self.V = V
        self.k = k
        self.batch_size = batch_size
        self.max_history = max(max_history, k)
        self.gen = torch.Generator(device=DEVICE)
        self.gen.manual_seed(seed + 4_000_003)
        self.history = torch.randint(
            0, V, (batch_size, self.max_history),
            generator=self.gen, device=DEVICE
        )
        for _ in range(warmup_steps):
            self.advance()

    @torch.no_grad()
    def draw_next(self):
        full_k = self.history[:, -self.k:]
        idx = context_to_index(full_k, self.V)
        probs = self.P[idx]
        return torch.multinomial(probs, 1, generator=self.gen).squeeze(1)

    @torch.no_grad()
    def advance(self):
        y = self.draw_next()
        self.history = torch.cat([self.history[:, 1:], y[:, None]], dim=1)
        return y

    @torch.no_grad()
    def sample(self, L):
        x = self.history[:, -L:].clone()
        y = self.draw_next()
        self.history = torch.cat([self.history[:, 1:], y[:, None]], dim=1)
        return x, y


## H* and H_L

For \(L<k\), the notebook estimates the Bayes-optimal truncated predictor

\[
q_L(y|c)=E[P(y|S)\mid C_L=c]
\]

from stationary full-state samples, while using the **true transition probabilities** rather than sampled next-token labels. For \(L\ge k\), \(H_L=H^*\) by the Markov property.


In [ ]:

@torch.no_grad()
def estimate_theory(P, V, k, max_L, n_samples=THEORY_SAMPLES, batch_size=THEORY_BATCH, seed=12345):
    max_history = max(max_L, k)
    sampler = PersistentMarkovSampler(P, V, k, batch_size, max_history, seed, warmup_steps=256)

    mix_sum = {}
    mix_count = {}
    for L in range(1, min(k, max_L + 1)):
        mix_sum[L] = torch.zeros((V**L, V), dtype=torch.float64, device=DEVICE)
        mix_count[L] = torch.zeros((V**L,), dtype=torch.float64, device=DEVICE)

    hstar_sum = 0.0
    seen = 0
    while seen < n_samples:
        b = min(batch_size, n_samples - seen)
        if b != batch_size:
            tmp = PersistentMarkovSampler(P, V, k, b, max_history, seed + seen + 1, warmup_steps=256)
            hist = tmp.history
        else:
            hist = sampler.history

        full = hist[:, -k:]
        p_full = P[context_to_index(full, V)].double()
        hstar_sum += (-(p_full * torch.log(p_full.clamp_min(1e-30))).sum(-1)).sum().item()

        for L in mix_sum:
            idx = context_to_index(hist[:, -L:], V)
            mix_sum[L].index_add_(0, idx, p_full)
            mix_count[L].index_add_(0, idx, torch.ones(b, dtype=torch.float64, device=DEVICE))

        if b == batch_size:
            sampler.advance()
        seen += b

    H_star = hstar_sum / seen
    q = {}
    for L in mix_sum:
        q[L] = mix_sum[L] / mix_count[L][:, None].clamp_min(1.0)

    sampler2 = PersistentMarkovSampler(P, V, k, batch_size, max_history, seed + 99191, warmup_steps=256)
    ce_sum = {L: 0.0 for L in q}
    seen = 0
    while seen < n_samples:
        b = min(batch_size, n_samples - seen)
        if b != batch_size:
            tmp = PersistentMarkovSampler(P, V, k, b, max_history, seed + 99191 + seen + 1, warmup_steps=256)
            hist = tmp.history
        else:
            hist = sampler2.history

        p_full = P[context_to_index(hist[:, -k:], V)].double()
        for L in q:
            idx = context_to_index(hist[:, -L:], V)
            q_batch = q[L][idx].clamp_min(1e-30)
            ce_sum[L] += (-(p_full * torch.log(q_batch)).sum(-1)).sum().item()

        if b == batch_size:
            sampler2.advance()
        seen += b

    out = {}
    for L in range(1, max_L + 1):
        H_L = H_star if L >= k else ce_sum[L] / seen
        out[L] = {
            "H_star": float(H_star),
            "H_L": float(H_L),
            "info_gap": float(H_L - H_star),
        }
    return out

def theory_path(V, k, seed, sparse_s, max_L):
    return THEORY_DIR / f"V{V}_k{k}_seed{seed}_sp{sparse_s or 0}_L{max_L}_N{THEORY_SAMPLES}.json"

def get_theory(V, k, seed, sparse_s=None, max_L=None, force=False):
    max_L = max_L or (k + EXTRA_CONTEXT)
    p = theory_path(V, k, seed, sparse_s, max_L)
    if p.exists() and not force:
        raw = json.load(open(p, "r", encoding="utf-8"))
        return {int(a): b for a, b in raw.items()}
    P = generate_kernel(V, k, seed, sparse_s)
    out = estimate_theory(P, V, k, max_L, THEORY_SAMPLES, THEORY_BATCH, seed + 1234)
    save_json(p, out)
    return out


In [ ]:

class TinyCausalTransformer(nn.Module):
    def __init__(self, V, max_context, d_model=256, n_layers=3, n_heads=8, dim_ff=512):
        super().__init__()
        self.token_emb = nn.Embedding(V, d_model)
        self.pos_emb = nn.Embedding(max_context, d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=dim_ff,
            dropout=0.0,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, V)
        self.max_context = max_context

    def forward(self, x):
        B, L = x.shape
        pos = torch.arange(L, device=x.device)[None, :]
        h = self.token_emb(x) + self.pos_emb(pos)
        mask = torch.triu(torch.ones((L, L), dtype=torch.bool, device=x.device), diagonal=1)
        h = self.encoder(h, mask=mask)
        return self.head(self.norm(h[:, -1, :]))

def nparams(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


In [ ]:

@dataclass
class RunConfig:
    experiment: str
    V: int
    k: int
    L: int
    seed: int
    sparse_s: Optional[int] = None
    steps: int = BASE_STEPS
    batch_size: int = BASE_BATCH
    lr: float = BASE_LR
    weight_decay: float = WEIGHT_DECAY
    d_model: int = BASE_DMODEL
    n_layers: int = BASE_LAYERS
    n_heads: int = BASE_HEADS
    dim_ff: int = BASE_FF
    eval_batch: int = EVAL_BATCH
    save_every: int = SAVE_EVERY

    def run_id(self):
        d = asdict(self)
        return (
            f"{self.experiment}_V{self.V}_k{self.k}_L{self.L}_seed{self.seed}"
            f"_sp{self.sparse_s or 0}_d{self.d_model}_ly{self.n_layers}_{stable_hash(d)}"
        )

def rdir(cfg):
    return RUNS_DIR / cfg.experiment / cfg.run_id()

def p_done(cfg): return rdir(cfg) / "DONE"
def p_ckpt(cfg): return rdir(cfg) / "checkpoint_last.pt"
def p_best(cfg): return rdir(cfg) / "checkpoint_best.pt"
def p_kernel(cfg): return rdir(cfg) / "kernel.pt"
def p_hist(cfg): return rdir(cfg) / "metrics.csv"
def p_final(cfg): return rdir(cfg) / "final_metrics.json"


In [ ]:

@torch.no_grad()
def evaluate(model, P, V, k, L, batch_size, seed):
    model.eval()
    sampler = PersistentMarkovSampler(P, V, k, batch_size, max(L, k), seed, warmup_steps=128)
    x, y = sampler.sample(L)
    with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=torch.cuda.is_available()):
        logits = model(x)
    return float(F.cross_entropy(logits.float(), y).item())

def save_ckpt(path, model, optimizer, scheduler, step, best_eval, history, cfg):
    payload = {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "step": step,
        "best_eval": best_eval,
        "history": history,
        "rng_state": get_rng_state(),
        "config": asdict(cfg),
    }
    tmp = Path(str(path) + ".tmp")
    torch.save(payload, tmp)
    os.replace(tmp, path)

def train_run(cfg, H_star=None, H_L=None, force=False):
    rd = rdir(cfg)
    rd.mkdir(parents=True, exist_ok=True)
    save_json(rd / "config.json", asdict(cfg))

    if p_done(cfg).exists() and not force:
        print("SKIP DONE:", cfg.run_id())
        return json.load(open(p_final(cfg), "r", encoding="utf-8"))

    set_seed(cfg.seed)

    if p_kernel(cfg).exists():
        P = torch.load(p_kernel(cfg), map_location=DEVICE)
    else:
        P = generate_kernel(cfg.V, cfg.k, cfg.seed, cfg.sparse_s).detach().cpu()
        torch.save(P, p_kernel(cfg))
        P = P.to(DEVICE)

    model = TinyCausalTransformer(
        cfg.V,
        max_context=max(cfg.L, cfg.k + EXTRA_CONTEXT),
        d_model=cfg.d_model,
        n_layers=cfg.n_layers,
        n_heads=cfg.n_heads,
        dim_ff=cfg.dim_ff,
    ).to(DEVICE)

    optimizer = AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=cfg.steps)

    start = 0
    best_eval = float("inf")
    history = []

    if p_ckpt(cfg).exists() and not force:
        ck = torch.load(p_ckpt(cfg), map_location=DEVICE)
        model.load_state_dict(ck["model"])
        optimizer.load_state_dict(ck["optimizer"])
        scheduler.load_state_dict(ck["scheduler"])
        start = int(ck["step"]) + 1
        best_eval = float(ck["best_eval"])
        history = ck.get("history", [])
        set_rng_state(ck.get("rng_state"))
        print("RESUME:", cfg.run_id(), "from", start)

    sampler = PersistentMarkovSampler(
        P, cfg.V, cfg.k, cfg.batch_size, max(cfg.L, cfg.k),
        cfg.seed + 888001, warmup_steps=CHAIN_WARMUP
    )

    model.train()
    t0 = time.time()

    for step in range(start, cfg.steps):
        x, y = sampler.sample(cfg.L)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=torch.cuda.is_available()):
            logits = model(x)
            loss = F.cross_entropy(logits.float(), y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        if step == 0 or (step + 1) % 100 == 0:
            history.append({
                "step": step + 1,
                "train_loss": float(loss.item()),
                "lr": float(scheduler.get_last_lr()[0]),
                "elapsed_s": time.time() - t0,
            })

        if (step + 1) % cfg.save_every == 0 or step + 1 == cfg.steps:
            ev = evaluate(model, P, cfg.V, cfg.k, cfg.L, cfg.eval_batch, cfg.seed + 77777 + step)
            history.append({
                "step": step + 1,
                "eval_loss": ev,
                "lr": float(scheduler.get_last_lr()[0]),
                "elapsed_s": time.time() - t0,
            })
            pd.DataFrame(history).to_csv(p_hist(cfg), index=False)

            if ev < best_eval:
                best_eval = ev
                save_ckpt(p_best(cfg), model, optimizer, scheduler, step, best_eval, history, cfg)
            save_ckpt(p_ckpt(cfg), model, optimizer, scheduler, step, best_eval, history, cfg)

            print(
                f"{cfg.experiment} V={cfg.V} k={cfg.k} L={cfg.L} seed={cfg.seed} "
                f"step={step+1}/{cfg.steps} train={loss.item():.4f} eval={ev:.4f}"
            )

    if p_best(cfg).exists():
        best = torch.load(p_best(cfg), map_location=DEVICE)
        model.load_state_dict(best["model"])

    test_loss = evaluate(model, P, cfg.V, cfg.k, cfg.L, cfg.eval_batch, cfg.seed + 999999)

    result = {
        **asdict(cfg),
        "run_id": cfg.run_id(),
        "run_dir": str(rd),
        "n_params": nparams(model),
        "test_loss": test_loss,
        "H_star": H_star,
        "H_L": H_L,
        "total_excess": None if H_star is None else test_loss - H_star,
        "info_gap": None if H_star is None or H_L is None else H_L - H_star,
        "model_gap": None if H_L is None else test_loss - H_L,
        "best_eval": best_eval,
        "elapsed_s": time.time() - t0,
    }

    save_json(p_final(cfg), result)
    p_done(cfg).write_text("completed\n", encoding="utf-8")
    return result


## Main recovery experiment

In [ ]:

main_results = []

if RUN_MAIN:
    for k in MAIN_KS:
        max_L = k + EXTRA_CONTEXT
        for seed in SEEDS:
            print(f"THEORY V={MAIN_V} k={k} seed={seed}")
            theory = get_theory(MAIN_V, k, seed, None, max_L)
            for L in range(1, max_L + 1):
                cfg = RunConfig(
                    experiment="main_recovery",
                    V=MAIN_V, k=k, L=L, seed=seed
                )
                main_results.append(
                    train_run(cfg, theory[L]["H_star"], theory[L]["H_L"])
                )

if main_results:
    pd.DataFrame(main_results).to_csv(CSV_DIR / "main_recovery_raw.csv", index=False)


In [ ]:

def load_completed(experiment=None):
    roots = [RUNS_DIR / experiment] if experiment else [p for p in RUNS_DIR.iterdir() if p.is_dir()]
    rows = []
    for root in roots:
        if not root.exists():
            continue
        for f in root.rglob("final_metrics.json"):
            if (f.parent / "DONE").exists():
                rows.append(json.load(open(f, "r", encoding="utf-8")))
    return pd.DataFrame(rows)

main_df = load_completed("main_recovery")

if len(main_df):
    agg = main_df.groupby(["V","k","L"], as_index=False).agg(
        test_loss_mean=("test_loss","mean"),
        test_loss_std=("test_loss","std"),
        H_star_mean=("H_star","mean"),
        H_L_mean=("H_L","mean"),
        total_excess_mean=("total_excess","mean"),
        total_excess_std=("total_excess","std"),
        info_gap_mean=("info_gap","mean"),
        model_gap_mean=("model_gap","mean"),
        n=("seed","count"),
    )
    agg.to_csv(CSV_DIR / "main_recovery_aggregated.csv", index=False)
    display(agg)

    plt.figure(figsize=(8,5.5))
    for k in sorted(agg.k.unique()):
        d = agg[agg.k == k].sort_values("L")
        plt.errorbar(d.L, d.total_excess_mean, yerr=d.total_excess_std.fillna(0),
                     marker="o", capsize=3, label=f"k={k}")
    plt.axhline(0, linewidth=1)
    plt.xlabel("Context length L")
    plt.ylabel("Excess cross-entropy (nats)")
    plt.title("Recovery curves")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / "main_recovery.png", dpi=300)
    plt.savefig(FIG_DIR / "main_recovery.pdf")
    plt.show()

    plt.figure(figsize=(8,5.5))
    for k in sorted(agg.k.unique()):
        d = agg[agg.k == k].sort_values("L")
        plt.plot(d.L, d.info_gap_mean, marker="o", label=f"Information gap k={k}")
        plt.plot(d.L, d.model_gap_mean, marker="x", linestyle="--", label=f"Model gap k={k}")
    plt.axhline(0, linewidth=1)
    plt.xlabel("Context length L")
    plt.ylabel("Gap (nats)")
    plt.title("Information vs model/optimization gap")
    plt.legend(ncol=2, fontsize=8)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "information_decomposition.png", dpi=300)
    plt.savefig(FIG_DIR / "information_decomposition.pdf")
    plt.show()

    pivot = agg.pivot(index="k", columns="L", values="total_excess_mean")
    plt.figure(figsize=(8,4))
    plt.imshow(pivot.values, aspect="auto", origin="lower")
    plt.colorbar(label="Mean excess cross-entropy")
    plt.xticks(range(len(pivot.columns)), pivot.columns)
    plt.yticks(range(len(pivot.index)), pivot.index)
    plt.xlabel("Context length L")
    plt.ylabel("Markov order k")
    plt.title("Phase diagram")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "phase_diagram.png", dpi=300)
    plt.savefig(FIG_DIR / "phase_diagram.pdf")
    plt.show()


## Capacity control

In [ ]:

CAPACITY = {
    "small": dict(d_model=128, n_layers=2, n_heads=4, dim_ff=256),
    "base":  dict(d_model=256, n_layers=3, n_heads=8, dim_ff=512),
    "large": dict(d_model=512, n_layers=4, n_heads=8, dim_ff=1024),
}

if RUN_CAPACITY:
    for seed in SEEDS:
        theory = get_theory(8, 4, seed, None, 4)
        for name, mc in CAPACITY.items():
            for L in [3, 4]:
                cfg = RunConfig(
                    experiment=f"capacity_{name}",
                    V=8, k=4, L=L, seed=seed,
                    steps=CAPACITY_STEPS,
                    **mc
                )
                train_run(cfg, theory[L]["H_star"], theory[L]["H_L"])


## Vocabulary robustness

In [ ]:

if RUN_VOCAB32:
    settings = [(32,2)]
    if RUN_VOCAB32_K3_EXT:
        settings.append((32,3))

    for V, k in settings:
        max_L = k + EXTRA_CONTEXT
        for seed in SEEDS:
            theory = get_theory(V, k, seed, None, max_L)
            for L in range(1, max_L + 1):
                steps = BASE_STEPS if k == 2 else max(BASE_STEPS, 12000)
                cfg = RunConfig(
                    experiment=f"vocab_V{V}_k{k}",
                    V=V, k=k, L=L, seed=seed,
                    steps=steps
                )
                train_run(cfg, theory[L]["H_star"], theory[L]["H_L"])


## Sparse-kernel robustness

In [ ]:

if RUN_SPARSE:
    for sparse_s in [2,4]:
        for k in MAIN_KS:
            max_L = k + EXTRA_CONTEXT
            for seed in SEEDS:
                theory = get_theory(8, k, seed, sparse_s, max_L)
                for L in range(1, max_L + 1):
                    cfg = RunConfig(
                        experiment=f"sparse_s{sparse_s}",
                        V=8, k=k, L=L, seed=seed,
                        sparse_s=sparse_s
                    )
                    train_run(cfg, theory[L]["H_star"], theory[L]["H_L"])


## Master export

In [ ]:

master = load_completed()
if len(master):
    sort_cols = [c for c in ["experiment","V","k","L","seed"] if c in master.columns]
    master = master.sort_values(sort_cols)
    master.to_csv(EXPORTS_DIR / "master_results.csv", index=False)
    print("Completed runs:", len(master))
    display(master.groupby("experiment").size().rename("completed_runs").to_frame())
    display(master.head())

print("Exports:", EXPORTS_DIR)


In [ ]:
from pathlib import Path
import zipfile

root = Path.cwd() / "Markov_Context_Paper" / "Part1"
output = Path.cwd() / "Part1_MAIN_3000_RESULTS.zip"

allowed_extensions = {
    ".csv",
    ".json",
    ".png",
    ".pdf",
    ".txt",
}

with zipfile.ZipFile(output, "w", zipfile.ZIP_DEFLATED) as z:
    for file in root.rglob("*"):
        if file.is_file() and file.suffix.lower() in allowed_extensions:
            z.write(file, file.relative_to(root))

print(f"Έτοιμο: {output.name}")
print(f"Μέγεθος: {output.stat().st_size / 1024**2:.2f} MB")

In [ ]:
# ============================================================
# FIX RNG RESTORE AFTER CUDA CHECKPOINT LOAD
# ============================================================

def set_rng_state(state):
    if state is None:
        return

    # Python RNG
    random.setstate(state["python"])

    # NumPy RNG
    np.random.set_state(state["numpy"])

    # PyTorch CPU RNG must be a CPU ByteTensor
    cpu_state = state["torch_cpu"]
    if isinstance(cpu_state, torch.Tensor):
        cpu_state = cpu_state.detach().cpu().to(torch.uint8)

    torch.set_rng_state(cpu_state)

    # CUDA RNG states
    if torch.cuda.is_available() and "torch_cuda" in state:
        cuda_states = state["torch_cuda"]

        fixed_cuda_states = []
        for s in cuda_states:
            if isinstance(s, torch.Tensor):
                s = s.detach().cpu().to(torch.uint8)
            fixed_cuda_states.append(s)

        torch.cuda.set_rng_state_all(fixed_cuda_states)

print("RNG restore fix loaded successfully.")

In [ ]:
# ============================================================
# K=4 REFINEMENT — ONLY L=4,5,6,7 — 12,000 STEPS
# 20 runs total
# ============================================================

REFINE_K = 4
REFINE_LS = [4, 5, 6, 7]
REFINE_SEEDS = [0, 1, 2, 3, 4]
REFINE_STEPS = 12_000

refinement_results = []

print("=" * 70)
print("K=4 REFINEMENT")
print(f"L values : {REFINE_LS}")
print(f"Seeds    : {REFINE_SEEDS}")
print(f"Steps    : {REFINE_STEPS}")
print(f"Runs     : {len(REFINE_LS) * len(REFINE_SEEDS)}")
print("=" * 70)

for seed in REFINE_SEEDS:

    # Same Markov source definition / theory calculation
    theory = get_theory(
        V=8,
        k=REFINE_K,
        seed=seed,
        sparse_s=None,
        max_L=max(REFINE_LS)
    )

    for L in REFINE_LS:

        print("\n" + "-" * 60)
        print(
            f"Starting refinement: "
            f"k={REFINE_K}, L={L}, seed={seed}"
        )
        print("-" * 60)

        cfg = RunConfig(
            experiment="k4_refinement_12k",

            V=8,
            k=REFINE_K,
            L=L,
            seed=seed,

            sparse_s=None,

            steps=REFINE_STEPS,

            # SAME architecture/training setup as main experiment
            batch_size=BASE_BATCH,
            lr=BASE_LR,
            weight_decay=WEIGHT_DECAY,

            d_model=BASE_DMODEL,
            n_layers=BASE_LAYERS,
            n_heads=BASE_HEADS,
            dim_ff=BASE_FF,

            eval_batch=EVAL_BATCH,
            save_every=SAVE_EVERY,
        )

        result = train_run(
            cfg,
            H_star=theory[L]["H_star"],
            H_L=theory[L]["H_L"],
        )

        refinement_results.append(result)


# ============================================================
# EXPORT RESULTS
# ============================================================

refine_df = pd.DataFrame(refinement_results)

refine_file = CSV_DIR / "k4_refinement_12k_raw.csv"
refine_df.to_csv(refine_file, index=False)

print("\n")
print("=" * 70)
print("REFINEMENT FINISHED")
print("=" * 70)
print(f"Saved to: {refine_file}")

display(refine_df)

In [ ]:
import json
import pandas as pd
from pathlib import Path

def load_completed(experiment=None):
    if experiment:
        roots = [RUNS_DIR / experiment]
    else:
        roots = [p for p in RUNS_DIR.iterdir() if p.is_dir()]

    rows = []

    for root in roots:
        if not root.exists():
            continue

        for f in root.rglob("final_metrics.json"):
            if (f.parent / "DONE").exists():
                with open(f, "r", encoding="utf-8") as handle:
                    rows.append(json.load(handle))

    return pd.DataFrame(rows)

print("load_completed restored.")

In [ ]:
refine_df = load_completed("k4_refinement_12k")

print("Completed refinement runs:", len(refine_df))

display(
    refine_df[
        ["k", "L", "seed", "steps",
         "test_loss", "H_star", "H_L",
         "total_excess", "info_gap", "model_gap"]
    ].sort_values(["L", "seed"])
)

In [ ]:
# ============================================================
# Monte Carlo uncertainty audit for H_L
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

AUDIT_SAMPLE_SIZES = [100_000, 250_000, 500_000, 1_000_000]
AUDIT_REPS = 20

# Main paper setting
AUDIT_V = 8
AUDIT_KS = [2, 3, 4]

rows = []

for k in AUDIT_KS:
    max_L = k - 1  # only L<k needs Monte Carlo H_L

    for source_seed in SEEDS:

        # SAME Markov kernel as main experiment
        P = generate_kernel(AUDIT_V, k, source_seed, sparse_s=None)

        for N in AUDIT_SAMPLE_SIZES:
            for rep in range(AUDIT_REPS):

                mc_seed = 900_000 + 10_000*k + 100*source_seed + rep

                theory = estimate_theory(
                    P=P,
                    V=AUDIT_V,
                    k=k,
                    max_L=max_L,
                    n_samples=N,
                    batch_size=THEORY_BATCH,
                    seed=mc_seed
                )

                for L in range(1, k):
                    rows.append({
                        "V": AUDIT_V,
                        "k": k,
                        "L": L,
                        "source_seed": source_seed,
                        "N": N,
                        "rep": rep,
                        "H_star": theory[L]["H_star"],
                        "H_L": theory[L]["H_L"],
                        "info_gap": theory[L]["info_gap"],
                    })

                print(
                    f"DONE k={k} source_seed={source_seed} "
                    f"N={N} rep={rep+1}/{AUDIT_REPS}"
                )

audit_df = pd.DataFrame(rows)

audit_file = CSV_DIR / "reviewer_R1_5_HL_uncertainty_raw.csv"
audit_df.to_csv(audit_file, index=False)

summary = (
    audit_df
    .groupby(["V", "k", "L", "N"], as_index=False)
    .agg(
        H_star_mean=("H_star", "mean"),
        H_star_sd=("H_star", "std"),
        H_L_mean=("H_L", "mean"),
        H_L_sd=("H_L", "std"),
        info_gap_mean=("info_gap", "mean"),
        info_gap_sd=("info_gap", "std"),
        n=("info_gap", "count"),
    )
)

summary["info_gap_ci95_halfwidth"] = (
    1.96 * summary["info_gap_sd"] / np.sqrt(summary["n"])
)

summary["relative_mc_uncertainty_pct"] = (
    100 * summary["info_gap_ci95_halfwidth"]
    / summary["info_gap_mean"].abs()
)

summary_file = CSV_DIR / "reviewer_R1_5_HL_uncertainty_summary.csv"
summary.to_csv(summary_file, index=False)

print("\nRAW:", audit_file)
print("SUMMARY:", summary_file)

display(summary)

print("\n=== N = 500,000 ONLY ===")
display(
    summary[summary["N"] == 500_000]
    .sort_values(["k", "L"])
)